In [ ]:
from __future__ import annotations
from pathlib import Path
import pickle, fsspec

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst



In [ ]:
fs = fsspec.filesystem("s3")
files = fs.ls("visionlab-members/amarvi/manifold-dynamics/timextime/ed_main")

# full s3 URIs
uris = [f"s3://{f}" for f in files]
local_paths = [vst.fetch(uri) for uri in uris]
rows = []
for fpath in local_paths:
    fpath = Path(fpath)

    with open(fpath, "rb") as f:
        ed = pickle.load(f)
    df_roi = ed.copy()
    rows.append(df_roi)

df = pd.concat(rows, ignore_index=True)

def zscore_group(g):
    mu = g.loc[g.condition == "random", "ED"].mean()
    sigma = g.loc[g.condition == "random", "ED"].std()
    g["ED_z"] = (g["ED"] - mu) / sigma
    return g

df = df.groupby(["roi"], group_keys=False).apply(zscore_group)

In [ ]:
df["target"].unique()

In [ ]:
def target_group(target):
    s = str(target)

    # "middle value" after splitting on "."
    parts = s.split(".")
    middle = parts[1] if len(parts) >= 3 else s

    if middle == "Unknown":
        return "Unknown"
    elif "F" in middle:
        return "F"
    elif "B" in middle:
        return "B"
    elif "O" in middle:
        return "O"
    else:
        return "Other"

df["target_group"] = df["target"].apply(target_group)
df = df[df["condition"] != "random"]

df.groupby(["condition", "target_group"])["ED_z"].describe()

In [ ]:
order = ["F", "B", "O", "Unknown", "Other"]
cond_order = ["global", "local"]
customp = ["#ffa4c7", "#5aa4d1"]

fig, ax = plt.subplots(1,1, figsize=(3,2))
sns.barplot(df, 
            x='target_group', 
            y='ED_z', 
            palette=customp, 
            hue='condition', 
            order=order, 
            hue_order=cond_order,
            errorbar="se",
            errwidth=0.5,
            ax=ax)

ax.set_xticklabels(["Face", "Body", "Object", "Unknown", "Other"])

ax.tick_params(axis="x", labelrotation=45)

ax.set_ylabel("ED (z-score)")
ax.set_ylim(ax.get_ylim())
ax.invert_yaxis()
# ax.legend(frameon=False, loc="upper center", ncol=2)
ax.legend()

sns.despine(ax=ax, trim=True, offset=5)
for label in ax.get_xticklabels():
    label.set_fontsize(12)

# individual dots
sns.stripplot(
    data=df,
    x='target_group',
    y='ED_z',
    hue='condition',
    order=order,
    hue_order=cond_order,
    dodge=True,           # separate by condition
    palette=["black"],
    alpha=0.2,
    size=3,
    legend=False,
    ax=ax
)
ax.set_xlabel("")

save_path = Path.home() / "Downloads" / f"ed_main.png"
# plt.savefig(save_path, transparent=False, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
target_value = "07.MF1.F"  # or whatever target you want

df_target = pd.concat(
    [r[r["roi"] == target_value] for r in rows],
    ignore_index=True
)
# optional: z-score
mu = df_target['ED'].mean()
sigma = df_target['ED'].std()
df_target["ED_z"] = (df_target["ED"] - mu) / sigma


order = ["global", "local", "random"]
fig, ax = plt.subplots(1,1, figsize=(2,2))
sns.boxplot(df_target, x="condition", y="ED_z", hue="condition", palette=["white"], order=order)
sns.stripplot(df_target, x="condition", y="ED_z", hue="condition", palette=["black"], alpha=0.05)

ax.set_ylabel("Effective\nDimensionality (ED)")

ax.set_xticklabels(["Global", "Local", "Random"])
ax.tick_params(axis="x", labelrotation=45)

ax.set_xlabel("")

sns.despine(ax=ax, trim=True, offset=5)
save_path = Path.home() / "Downloads" / f"ed_{target_value}.png"
plt.savefig(save_path, transparent=False, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# sns.histplot(df_target, x="ED", hue="condition")
fig, ax = plt.subplots(1,1, figsize=(2,1.5))

sns.barplot(df_target, x="condition", y="ED_z", hue="condition", order=["global", "local"])
sns.despine(ax=ax, trim=True, offset=5)
for label in ax.get_xticklabels():
    label.set_fontsize(12)

ax.set_xlabel("")
ax.set_ylabel("ED (z-score)")
ax.set_ylim(ax.get_ylim())
ax.invert_yaxis()

ax.set_xticklabels(["Global", "Local"])
# ax.tick_params(axis="x", labelrotation=45)

ax.set_title("MF7")

save_path = Path.home() / "Downloads" / f"{target_value}_zbars.svg"
plt.savefig(save_path, transparent=False, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sns.stripplot(
    data=df[df.condition != "random"],
    x="ED_z",
    y="target",   # or target
    hue="condition",
    dodge=True,
)

plt.axvline(0, linestyle="--", color="black")
plt.xlabel("ED (z-score vs random)")
plt.show()

In [ ]:
feature_layers = ['classifier.2']  # add more if needed
s3_base = f'{pth.SAVEDIR}/neighbors/{'_'.join(feature_layers)}/{target_value}'
neighbor_local = vst.fetch(f'{s3_base}.pkl')

with open(neighbor_local, 'rb') as f:
    df_out = pickle.load(f)

# reshape to long format because wide tables are how plots start fights
ed_topk = float(np.squeeze(df_out['ED_topk'].unique()))

df_target = pd.DataFrame({
    'condition': ['global', 'local', 'random'],
    'ED': [
        ed_topk,
        df_out['ED_neighbor'].to_numpy(),
        df_out['ED_random'].to_numpy(),
    ],
})

df_target = df_target.explode('ED', ignore_index=True)
df_target['ED'] = pd.to_numeric(df_target['ED'])

# optional: z-score across all conditions for this target
mu = df_target['ED'].mean()
sigma = df_target['ED'].std()
df_target['ED_z'] = (df_target['ED'] - mu) / sigma

order = ['global', 'local', 'random']

fig, ax = plt.subplots(1, 1, figsize=(2, 2))

sns.boxplot(
    data=df_target,
    x='condition',
    y='ED_z',
    hue='condition',
    order=order,
    palette=['white', 'white', 'white'],
    ax=ax,
)

sns.stripplot(
    data=df_target,
    x='condition',
    y='ED_z',
    hue='condition',
    order=order,
    palette=['black', 'black', 'black'],
    alpha=0.08,
    dodge=False,
    ax=ax,
)

# remove duplicated legends from hue-on-hue chaos
if ax.legend_ is not None:
    ax.legend_.remove()

ax.set_ylabel('Effective\nDimensionality (ED)')
ax.set_xlabel('')
ax.set_title('')

ax.set_xticklabels(['Top-k', 'AlexNet\nfc6', 'Random'])
ax.tick_params(axis='x', labelrotation=45)

sns.despine(fig=fig, trim=True, offset=5)

save_path = Path.home() / 'Downloads' / f'ed_alexnet_{target_value}.svg'
plt.savefig(save_path, transparent=False, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
target_value = '07.MF1.F'

s3_base = f'{pth.SAVEDIR}/timextime/local_scale_ed/{target_value}'
other_roi_local = vst.fetch(f'{s3_base}.pkl')

with open(other_roi_local, 'rb') as f:
    out = pickle.load(f)

df_out = out['df_out'].copy()

# optional: z-score within target across all conditions
mu = df_out['ED'].mean()
sigma = df_out['ED'].std()
df_out['ED_z'] = (df_out['ED'] - mu) / sigma

order = ['top-k', 'alternative', 'random']

fig, ax = plt.subplots(1, 1, figsize=(2, 2))

sns.boxplot(
    data=df_out,
    x='condition',
    y='ED_z',
    hue='condition',
    order=order,
    palette=['white', 'white', 'white'],
    ax=ax,
)

sns.stripplot(
    data=df_out,
    x='condition',
    y='ED_z',
    hue='condition',
    order=order,
    palette=['black', 'black', 'black'],
    alpha=0.05,
    dodge=False,
    ax=ax,
)

# duplicated legends are a timeless tradition
if ax.legend_ is not None:
    ax.legend_.remove()

ax.set_title('')
ax.set_xlabel('')
ax.set_ylabel('Effective\nDimensionality (ED)')
ax.set_xticklabels(['Top-k', 'Other patch', 'Random'])
ax.tick_params(axis='x', labelrotation=35)

sns.despine(fig=fig, trim=True, offset=5)

save_path = Path.home() / 'Downloads' / f'patch_ed_{target_value}.svg'
plt.savefig(save_path, transparent=False, bbox_inches='tight', dpi=300)
plt.show()